# RetainIQ — Phase 3.4: Views, Indexes, Performance & Documentation

## Objective

I finish the MySQL layer by creating reusable views, adding targeted indexes, checking query plans, and documenting how SQL will feed Power BI and the AI/RAG workflow.

## 1. Connect to MySQL

In [1]:
from getpass import getpass
import mysql.connector
MYSQL_CONFIG={"host":"localhost","port":3306,"user":"retainiq_user","password":getpass("Enter MySQL password for retainiq_user: "),"database":"retainiq"}

def execute_sql(sql):
    connection=mysql.connector.connect(**MYSQL_CONFIG); cursor=connection.cursor()
    try:
        for statement in [s.strip() for s in sql.split(";") if s.strip()]: cursor.execute(statement)
        connection.commit()
    except Exception:
        connection.rollback(); raise
    finally:
        cursor.close(); connection.close()

## 2. Create Reusable Reporting Views

In [2]:
views_sql="""
CREATE OR REPLACE VIEW vw_customer_360 AS
SELECT f.customer_id,f.customer_status,f.churn_label,f.churn_score,f.cltv,f.monthly_charge,f.total_revenue,f.satisfaction_score,a.contract,a.payment_method,a.offer,s.internet_type,s.premium_tech_support,d.gender,d.age,d.senior_citizen,l.state,l.city,l.latitude,l.longitude
FROM fact_customer_status f
JOIN dim_account a ON f.customer_id=a.customer_id
JOIN dim_services s ON f.customer_id=s.customer_id
JOIN dim_demographics d ON f.customer_id=d.customer_id
JOIN dim_location l ON f.customer_id=l.customer_id;

CREATE OR REPLACE VIEW vw_retention_summary AS
SELECT COUNT(*) total_customers,SUM(churn_label='Yes') churned_customers,ROUND(100*SUM(churn_label='Yes')/COUNT(*),2) churn_rate_pct,SUM(CASE WHEN churn_label='Yes' THEN total_revenue ELSE 0 END) revenue_at_risk,ROUND(AVG(cltv),2) avg_cltv,ROUND(AVG(satisfaction_score),2) avg_satisfaction
FROM fact_customer_status;

CREATE OR REPLACE VIEW vw_contract_performance AS
SELECT a.contract,COUNT(*) customers,SUM(f.churn_label='Yes') churned_customers,ROUND(100*SUM(f.churn_label='Yes')/COUNT(*),2) churn_rate_pct,SUM(CASE WHEN f.churn_label='Yes' THEN f.total_revenue ELSE 0 END) revenue_at_risk,ROUND(AVG(f.cltv),2) avg_cltv
FROM fact_customer_status f JOIN dim_account a ON f.customer_id=a.customer_id GROUP BY a.contract;

CREATE OR REPLACE VIEW vw_revenue_at_risk AS
SELECT f.customer_id,f.cltv,f.total_revenue,f.churn_score,f.satisfaction_score,a.contract,a.offer,s.internet_type
FROM fact_customer_status f JOIN dim_account a ON f.customer_id=a.customer_id JOIN dim_services s ON f.customer_id=s.customer_id
WHERE f.churn_label='Yes';
"""
execute_sql(views_sql)
print("Reporting views created.")

Reporting views created.


## 3. Query the Reporting Views

In [3]:
import pandas as pd
connection=mysql.connector.connect(**MYSQL_CONFIG); cursor=connection.cursor(dictionary=True)
try:
    cursor.execute("SELECT * FROM vw_retention_summary"); retention_summary=pd.DataFrame(cursor.fetchall())
    cursor.execute("SELECT * FROM vw_contract_performance ORDER BY churn_rate_pct DESC"); contract_performance=pd.DataFrame(cursor.fetchall())
finally:
    cursor.close(); connection.close()
retention_summary, contract_performance

(   total_customers churned_customers churn_rate_pct revenue_at_risk avg_cltv  \
 0             7043              1869          26.54      3684459.82  4400.30   
 
   avg_satisfaction  
 0             3.24  ,
          contract  customers churned_customers churn_rate_pct revenue_at_risk  \
 0  Month-to-Month       3610              1655          45.84      2490105.85   
 1        One Year       1550               166          10.71       858489.80   
 2        Two Year       1883                48           2.55       335864.17   
 
   avg_cltv  
 0  4137.47  
 1  4500.57  
 2  4821.64  )

## 4. Add Targeted Indexes

In [4]:
index_sql="""
CREATE INDEX idx_fact_churn_label ON fact_customer_status(churn_label);
CREATE INDEX idx_fact_cltv ON fact_customer_status(cltv);
CREATE INDEX idx_fact_satisfaction ON fact_customer_status(satisfaction_score);
CREATE INDEX idx_account_contract ON dim_account(contract);
CREATE INDEX idx_account_payment_method ON dim_account(payment_method);
CREATE INDEX idx_services_internet_type ON dim_services(internet_type);
CREATE INDEX idx_location_state_city ON dim_location(state,city);
"""
execute_sql(index_sql)
print("Targeted indexes created.")

Targeted indexes created.


I only index fields I expect to filter, group, join, or sort on regularly. I avoid adding indexes to every column because indexes have storage and write-maintenance costs.

## 5. Inspect a Query Plan

In [5]:
explain="""EXPLAIN SELECT a.contract,COUNT(*) customers,SUM(f.churn_label='Yes') churned_customers FROM fact_customer_status f JOIN dim_account a ON f.customer_id=a.customer_id GROUP BY a.contract;"""
connection=mysql.connector.connect(**MYSQL_CONFIG); cursor=connection.cursor(dictionary=True)
try:
    cursor.execute(explain); explain_result=pd.DataFrame(cursor.fetchall())
finally:
    cursor.close(); connection.close()
explain_result

,id,select_type,table,partitions,type,possible_keys,key,key_len,ref,rows,filtered,Extra
0,1,SIMPLE,a,None,index,"PRIMARY,idx_account_contract",idx_account_contract,123,None,7280,100.0,Using index
1,1,SIMPLE,f,None,eq_ref,PRIMARY,PRIMARY,82,retainiq.a.customer_id,1,100.0,None


I use `EXPLAIN` to inspect how MySQL plans the query. I only claim a performance improvement after measuring the plan or execution time.

## 6. SQL Capability Checklist

In [6]:
skills=pd.DataFrame({"Capability":["DDL","DML","DQL","JOINs","Aggregation","CASE","CTEs","Window functions","Views","Indexes","EXPLAIN","Primary keys","Foreign keys","Validation / reconciliation"],"RetainIQ Use":["Database and table creation","Staging and table population","Business analysis","Fact-to-dimension joins","Churn / revenue / segment analysis","CLTV segmentation","High-value customer analysis","Ranking and contract revenue share","Reusable reporting datasets","Targeted analytical indexes","Query-plan inspection","Customer uniqueness","Referential integrity","7,043 / 1,869 / 26.5% checks"]})
skills

,Capability,RetainIQ Use
0,DDL,Database and table creation
1,DML,Staging and table population
2,DQL,Business analysis
3,JOINs,Fact-to-dimension joins
4,Aggregation,Churn / revenue / segment analysis
5,CASE,CLTV segmentation
6,CTEs,High-value customer analysis
7,Window functions,Ranking and contract revenue share
8,Views,Reusable reporting datasets
9,Indexes,Targeted analytical indexes


## 7. MySQL → Power BI

```text
MySQL
  │
  ├── vw_retention_summary
  ├── vw_contract_performance
  ├── vw_revenue_at_risk
  └── vw_customer_360
          │
          ▼
       Power BI
```

I keep reusable business logic in MySQL and let Power BI focus on interactive reporting.

## 8. MySQL → AI / RAG

```text
MySQL
   ↓
Validated analytical queries
   ↓
Business findings
   ↓
RAG knowledge base
   ↓
LLM
   ↓
Grounded retention insights
```

The AI layer should explain validated analytical outputs rather than replacing the database.

# Phase 3 Final Conclusion

I have built the complete MySQL analytical layer for RetainIQ.

I now have:
- a customer-level star schema
- staging and direct MySQL loading through Python
- fact and dimension tables
- keys and constraints
- validation queries
- business-analysis SQL
- joins and aggregation
- `CASE`
- CTEs
- window functions
- Customer 360
- reusable views
- targeted indexes
- `EXPLAIN`

### Completion gate
I will mark Phase 3 complete only after MySQL returns **7,043 customers, 1,869 churned customers, 26.5% churn, 0 duplicate fact keys, and 0 orphaned dimension rows.**

**Next:** Phase 4 — EDA & Statistical Modeling.